# WiTA v2 — Self-Contained Air-Writing Recognition (Kaggle T4)

**No `git clone` required.** All code is in this package.

| | |
|---|---|
| GPU | NVIDIA T4 (16 GB VRAM) |
| Encoder | 3-D ResNet (r3d / mc3 / r2plus1d) |
| Decoder | CTC + Transformer Attention (hybrid) |
| Target | val CER < 0.22 within 40 epochs |

**Pipeline:** install → setup → stream data → build model → train → evaluate → export

## Cell 1 — Install dependencies

In [ ]:
%%capture
!pip install editdistance huggingface_hub hgtk --quiet

# ── Copy wita_v2 package into Python path ────────────────────────────
# Upload the wita_v2/ folder as a Kaggle Dataset called 'wita-v2-package'.
# It will appear at /kaggle/input/wita-v2-package/wita_v2/
import sys, shutil, os

PKG_SRC = '/kaggle/input/wita-v2-package/wita_v2'
PKG_DST = '/kaggle/working/wita_v2'

if os.path.exists(PKG_SRC):
    if os.path.exists(PKG_DST):
        shutil.rmtree(PKG_DST)
    shutil.copytree(PKG_SRC, PKG_DST)
    print(f'Package copied from {PKG_SRC}')
elif not os.path.exists(PKG_DST):
    print('WARNING: wita_v2 package not found. '
          'Upload it as a Kaggle dataset or copy manually to /kaggle/working/wita_v2')

if PKG_DST not in sys.path:
    sys.path.insert(0, '/kaggle/working')
print('sys.path[0]:', sys.path[0])


## Cell 2 — Imports & Config

In [ ]:
import os, sys, logging, random
import numpy as np
import torch

os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
os.makedirs('/kaggle/working/logs',        exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-7s  %(name)s — %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler('/kaggle/working/logs/run.log'),
    ]
)

# ── Build Config ─────────────────────────────────────────────────────
from wita_v2.configs.default import (
    Config, DataConfig, AugConfig, EncoderConfig,
    RecurrentConfig, AttnDecoderConfig, TrainConfig
)

cfg = Config(
    data=DataConfig(
        hf_repo_id  = 'yewon816/WiTA',
        lang        = 'english',
        max_zips    = None,       # ← set to 2 for a smoke test
        img_size    = 112,
        max_frames  = 64,
    ),
    aug=AugConfig(),              # sensible defaults
    encoder=EncoderConfig(
        arch           = 'r3d',
        num_res_layers = 1,
        out_dim        = 256,
        pooling        = 'average',
        pretrained     = False,   # True to init from torchvision r3d_18
    ),
    recurrent=RecurrentConfig(
        arch        = 'lstm',     # 'lstm' | 'gru' | 'transformer' | 'none'
        hidden_size = 256,
        num_layers  = 2,
    ),
    attn=AttnDecoderConfig(
        n_layers = 4, n_heads = 8, ff_dim = 2048,
    ),
    train=TrainConfig(
        num_epochs      = 40,
        batch_size      = 4,      # micro-batch; effective = 4 × 4 = 16
        accum_steps     = 4,
        lr              = 3e-4,
        num_workers     = 2,
        optimizer       = 'adamw',
        scheduler       = 'onecycle',
        val_limit       = 50,     # batches per fast val pass (None = full)
        lambda_ctc_start= 0.5,
        lambda_ctc_min  = 0.2,
        label_smoothing = 0.1,
        checkpoint_dir  = '/kaggle/working/checkpoints',
        resume_path     = None,   # ← '/kaggle/working/checkpoints/latest.pt' to resume
        save_frequency  = 5,
        seed            = 42,
    ),
).build()   # finalises vocab indices

# Reproducibility
random.seed(cfg.train.seed)
np.random.seed(cfg.train.seed)
torch.manual_seed(cfg.train.seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

print(f'Device          : {cfg.device}')
print(f'CTC  vocab size : {cfg.vocab.ctc_vocab_size}  (blank=0, chars 1..{len(cfg.vocab.chars)})')
print(f'Attn vocab size : {cfg.vocab.attn_vocab_size}  '
      f'(SOS={cfg.vocab.sos_idx} EOS={cfg.vocab.eos_idx} PAD={cfg.vocab.pad_idx})')
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


## Cell 3 — Vocabulary sanity check

In [ ]:
from wita_v2.datasets.vocab import make_converter

converter = make_converter(cfg.data.lang)
print('Alphabet length:', len(converter.alphabet))

for word in ['hello', 'mississippi', 'book']:
    enc, length = converter.encode(word)
    dec = converter.decode(enc, length)
    match = '✓' if dec == word else f'✗ got "{dec}"'
    print(f'  "{word}" → {enc.tolist()} → "{dec}"  {match}')


## Cell 4 — Stream & index dataset from HuggingFace

In [ ]:
from wita_v2.datasets.dataset import stream_and_index

# Downloads ZIPs one-at-a-time, reads frames into RAM as raw bytes,
# deletes each ZIP immediately — never fills the disk with PNG files.
samples = stream_and_index(cfg)
print(f'Total samples indexed: {len(samples)}')

# Quick peek at one sample
frame_bytes, label = samples[0]
print(f'Sample 0: label="{label}"  frames={len(frame_bytes)}')
print(f'Frame[0] size: {len(frame_bytes[0])} bytes')


## Cell 5 — Build DataLoaders

In [ ]:
from wita_v2.datasets.dataset import make_dataloaders

train_loader, val_loader = make_dataloaders(cfg, samples, converter)
print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')

# Sanity check one batch
clips, labels, input_lens, label_lens = next(iter(train_loader))
print(f'clips      : {clips.shape}  {clips.dtype}')    # [B, T, C, H, W]
print(f'labels     : {labels.shape} {labels.dtype}')   # [B, L]
print(f'input_lens : {input_lens.tolist()}')
print(f'label_lens : {label_lens.tolist()}')

# Decode a label to verify
gt = converter.decode(labels[0, :label_lens[0].item()].int(),
                       torch.IntTensor([label_lens[0].item()]))
print(f'GT word[0] : "{gt}"')


## Cell 6 — Build model

In [ ]:
from wita_v2.models.hybrid_model import build_model
from wita_v2.training.losses import prepare_attn_targets

model = build_model(cfg).to(cfg.device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total:,}')
print(f'Trainable params: {trainable:,}')
print()
print('  Encoder  :', cfg.encoder.arch.upper(), '— out_dim', cfg.encoder.out_dim)
print('  Recurrent:', cfg.recurrent.arch.upper(),
      '— hidden', cfg.recurrent.hidden_size, '× 2 (bi)')
print('  CTC head  → vocab', cfg.vocab.ctc_vocab_size)
print('  Attn head → vocab', cfg.vocab.attn_vocab_size,
      f'({cfg.attn.n_layers}L × {cfg.attn.n_heads}H)')

# Forward sanity check (no OOM)
clips_b  = clips.to(cfg.device)
labels_b = labels.to(cfg.device)
il_b     = input_lens.to(cfg.device)
ll_b     = label_lens.to(cfg.device)
tgt_in, tgt_out, tgt_pad = prepare_attn_targets(labels_b, ll_b, cfg.vocab)

with torch.no_grad():
    ctc_out, attn_out = model(clips_b, il_b, tgt_in, tgt_pad)
print(f'\nCTC  output : {ctc_out.shape}   (T, B, {cfg.vocab.ctc_vocab_size})')
print(f'Attn output : {attn_out.shape}  (B, L+1, {cfg.vocab.attn_vocab_size})')

if torch.cuda.is_available():
    print(f'VRAM after forward: {torch.cuda.memory_allocated()/1e9:.2f} GB')
print('Forward pass OK ✓')


## Cell 7 — Train

Features enabled:
- **Mixed precision** (AMP) — halves VRAM bandwidth on T4
- **Gradient accumulation** — effective batch = `batch_size × accum_steps = 16`
- **Lambda annealing** — CTC weight 0.5 → 0.2 over epochs
- **Checkpointing** — `latest.pt` every epoch + `best.pt` on CER improvement
- **Resume** — set `cfg.train.resume_path` to continue after a crash
- **VAL_LIMIT** — fast 50-batch validation pass each epoch

In [ ]:
from wita_v2.training.trainer import train

best_model = train(
    model        = model,
    train_loader = train_loader,
    val_loader   = val_loader,
    converter    = converter,
    cfg          = cfg,
)
print('Training complete.')


## Cell 8 — Final evaluation (full val set, both decoders)

In [ ]:
from wita_v2.evaluation.evaluator import evaluate_cer, print_sample_table

print('Running full validation (no batch limit) …')

cer_ctc, _ = evaluate_cer(
    best_model, val_loader, converter, cfg,
    decode_mode='ctc', max_batches=None,
)
print(f'Final Val CER (CTC  greedy) : {cer_ctc:.4f}')

cer_attn, _ = evaluate_cer(
    best_model, val_loader, converter, cfg,
    decode_mode='attn', max_batches=None,
)
print(f'Final Val CER (Attn greedy) : {cer_attn:.4f}')

print_sample_table(
    best_model, val_loader, converter, cfg,
    epoch=cfg.train.num_epochs, max_batches=None,
)

print()
print('─' * 50)
print('Reference baselines')
print('  Original WiTA (3D-ResNet + CTC) : ~0.30')
print('  + Augmentation only             : ~0.24–0.26')
print('  + Hybrid CTC+Attn only          : ~0.22–0.25')
print('  Phase 1 target (both)           : ~0.18–0.22')
print(f'  This run (CTC / Attn)           : {cer_ctc:.4f} / {cer_attn:.4f}')


## Cell 9 — Export for Phase 2

In [ ]:
import os, torch

export_path = os.path.join(cfg.train.checkpoint_dir, 'phase1_export.pt')
torch.save({
    'model_state_dict': best_model.state_dict(),
    'ctc_vocab_size':   cfg.vocab.ctc_vocab_size,
    'attn_vocab_size':  cfg.vocab.attn_vocab_size,
    'lang':             cfg.data.lang,
    'encoder_arch':     cfg.encoder.arch,
    'encoder_dim':      cfg.encoder.out_dim,
    'val_cer_ctc':      cer_ctc,
    'val_cer_attn':     cer_attn,
}, export_path)

sz = os.path.getsize(export_path) / 1e6
print(f'Model exported → {export_path}  ({sz:.1f} MB)')
print('Ready for Phase 2 (Video Swin backbone, CLIP re-ranking).')


## Cell 10 — Optional: swap config for a quick ablation

Re-run from Cell 6 with any of these config tweaks:

In [ ]:
# ── Ablation: no recurrent head ──────────────────────────────────────
# cfg.recurrent.arch = 'none'

# ── Ablation: GRU instead of LSTM ────────────────────────────────────
# cfg.recurrent.arch = 'gru'

# ── Ablation: deeper ResNet (R3D-18) ─────────────────────────────────
# cfg.encoder.num_res_layers = 2

# ── Ablation: CTC-only (lambda_ctc = 1.0) ────────────────────────────
# cfg.train.lambda_ctc_start = 1.0
# cfg.train.lambda_ctc_min   = 1.0

# ── Ablation: no augmentation ─────────────────────────────────────────
# from wita_v2.configs.default import AugConfig
# cfg.aug = AugConfig(
#     mirror_prob=0, rotation_deg=0, brightness=0, contrast=0,
#     saturation=0, hue=0, grayscale_prob=0,
#     drop_frames_prob=0, temporal_crop_ratio=(1.0, 1.0),
# )
print('Edit this cell and re-run from Cell 6 to run ablations.')
